# Generating NER dataset

In [1]:
from src.load_ner_data import save_dataset

In [2]:
save_dataset()

Dataset saved to: ./src/data/text_data/ner_dataset.csv
Total samples: 5000


In [21]:
import torch

import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

In [4]:
df = pd.read_csv("./src/data/text_data/ner_dataset.csv")
df.head()

,tokens,ner_tags
0,I think it's a cow,O O O O B-ANIMAL
1,That might be a butterfly,O O O O B-ANIMAL
2,Could this be a squirrel?,O O O O B-ANIMAL
3,Could this be a spider?,O O O O B-ANIMAL
4,Maybe a dog,O O B-ANIMAL


# Data Preprocessing and tokenization

In [5]:
def preprocess(df):
    data = []

    for _, row in df.iterrows():
        tokens = row["tokens"].split()
        labels = row["ner_tags"].split()

        data.append({
            "tokens": tokens,
            "ner_tags": labels
        })

    return data

data = preprocess(df)
dataset = Dataset.from_list(data)

In [6]:
label_list = ["O", "B-ANIMAL"]

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

or the Named Entity Recognition (NER) task, we use **DistilBERT** *(https://huggingface.co/distilbert/distilbert-base-uncased)*.

In [7]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [8]:
def tokenize_and_align_labels(example):
    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True
    )

    word_ids = tokenized.word_ids()
    labels = []
    prev_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != prev_word_idx:
            labels.append(label2id[example["ner_tags"][word_idx]])
        else:
            labels.append(-100)

        prev_word_idx = word_idx

    tokenized["labels"] = labels
    return tokenized

dataset = dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

# Data train / val split and model train / evaluation

In [9]:
dataset = dataset.train_test_split(test_size=0.1)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

In [10]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
training_args = TrainingArguments(
    output_dir="./src/artifacts/ner_model",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_steps=50,
    report_to=[],
    disable_tqdm=True
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

In [17]:
trainer.train()

{'loss': '7.38e-06', 'grad_norm': '5.025e-05', 'learning_rate': '4.71e-05', 'epoch': '0.1773'}
{'loss': '1.613e-06', 'grad_norm': '3.137e-05', 'learning_rate': '4.415e-05', 'epoch': '0.3546'}
{'loss': '1.088e-06', 'grad_norm': '2.032e-05', 'learning_rate': '4.119e-05', 'epoch': '0.5319'}
{'loss': '7.806e-07', 'grad_norm': '1.56e-05', 'learning_rate': '3.824e-05', 'epoch': '0.7092'}
{'loss': '6.017e-07', 'grad_norm': '1.178e-05', 'learning_rate': '3.528e-05', 'epoch': '0.8865'}
{'loss': '4.934e-07', 'grad_norm': '1.058e-05', 'learning_rate': '3.233e-05', 'epoch': '1.064'}
{'loss': '4.137e-07', 'grad_norm': '8.351e-06', 'learning_rate': '2.937e-05', 'epoch': '1.241'}
{'loss': '3.574e-07', 'grad_norm': '7.856e-06', 'learning_rate': '2.642e-05', 'epoch': '1.418'}
{'loss': '3.147e-07', 'grad_norm': '7.334e-06', 'learning_rate': '2.346e-05', 'epoch': '1.596'}
{'loss': '2.818e-07', 'grad_norm': '5.995e-06', 'learning_rate': '2.051e-05', 'epoch': '1.773'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '2.561e-07', 'grad_norm': '5.721e-06', 'learning_rate': '1.755e-05', 'epoch': '1.95'}
{'loss': '2.376e-07', 'grad_norm': '5.608e-06', 'learning_rate': '1.46e-05', 'epoch': '2.128'}
{'loss': '2.197e-07', 'grad_norm': '5.078e-06', 'learning_rate': '1.164e-05', 'epoch': '2.305'}
{'loss': '2.104e-07', 'grad_norm': '5.32e-06', 'learning_rate': '8.688e-06', 'epoch': '2.482'}
{'loss': '2.016e-07', 'grad_norm': '4.7e-06', 'learning_rate': '5.733e-06', 'epoch': '2.66'}
{'loss': '1.967e-07', 'grad_norm': '4.411e-06', 'learning_rate': '2.778e-06', 'epoch': '2.837'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '62.05', 'train_samples_per_second': '217.6', 'train_steps_per_second': '13.63', 'train_loss': '8.761e-07', 'epoch': '3'}


TrainOutput(global_step=846, training_loss=8.760981993058539e-07, metrics={'train_runtime': 62.0485, 'train_samples_per_second': 217.572, 'train_steps_per_second': 13.634, 'train_loss': 8.760981993058539e-07, 'epoch': 3.0})

In [18]:
trainer.evaluate()

{'eval_loss': '1.204e-07', 'eval_runtime': '0.8403', 'eval_samples_per_second': '595', 'eval_steps_per_second': '38.08', 'epoch': '3'}


{'eval_loss': 1.2035556551381887e-07,
 'eval_runtime': 0.8403,
 'eval_samples_per_second': 595.013,
 'eval_steps_per_second': 38.081,
 'epoch': 3.0}

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

id2label = {v: k for k, v in label2id.items()}

def extract_animal(text, model, tokenizer):
    tokens = text.split()

    inputs = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )

    word_ids = inputs.word_ids()

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=2)

    predicted_labels = []
    previous_word_idx = None

    for idx, word_idx in enumerate(word_ids):
        if word_idx is None or word_idx == previous_word_idx:
            continue

        label_id = predictions[0][idx].item()
        label = id2label[label_id]

        predicted_labels.append((tokens[word_idx], label))
        previous_word_idx = word_idx

    animals = [word for word, label in predicted_labels if label != "O"]

    return animals

In [28]:
text = "I think there is a dog in the picture"

extract_animal(text, model, tokenizer)

['dog']